In [8]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

import flwr as fl

print("TensorFlow Version:", tf.__version__)
print("Flower (flwr) Version:", fl.__version__)

TensorFlow Version: 2.21.0
Flower (flwr) Version: 1.34.0


Node 1 (Iyin-Ekiti)   - Tree Crops & Berries : 0 classes
Node 2 (Ado-Ekiti)    - Grains & Citrus      : 0 classes
Node 3 (Ikere-Ekiti)  - Nightshades & Herbs  : 0 classes
Total Partitioned Classes                     : 0 / 3


### Multi-Farm Client Data Partitioning

In [2]:
# Load tabular training dataset
X_train = np.load('data/processed/X_train_tab.npy')
y_train = np.load('data/processed/y_train_tab.npy')

# Divide dataset into 3 simulated farm client nodes
NUM_CLIENTS = 3
client_data = []

X_splits = np.array_split(X_train, NUM_CLIENTS)
y_splits = np.array_split(y_train, NUM_CLIENTS)

for i in range(NUM_CLIENTS):
    client_data.append((X_splits[i], y_splits[i]))
    print(f"Farm Node {i+1} local dataset shape: {X_splits[i].shape}")

Farm Node 1 local dataset shape: (8522, 21)
Farm Node 2 local dataset shape: (8521, 21)
Farm Node 3 local dataset shape: (8521, 21)


### Local Model Architecture & Flower Client Definition

In [3]:
def build_federated_model(input_dim):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

class FarmNodeClient(fl.client.NumPyClient):
    def __init__(self, x_train, y_train, input_dim):
        self.x_train = x_train
        self.y_train = y_train
        self.model = build_federated_model(input_dim)

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)
        self.model.fit(self.x_train, self.y_train, epochs=3, batch_size=32, verbose=0)
        return self.model.get_weights(), len(self.x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        loss, accuracy = self.model.evaluate(self.x_train, self.y_train, verbose=0)
        return float(loss), len(self.x_train), {"accuracy": float(accuracy)}

### Run Federated Averaging (FedAvg) Simulation

In [6]:
import sys
import numpy as np

# Differential Privacy parameters
NOISE_SCALE = 0.01  # Calibrated noise scale for privacy protection

def apply_differential_privacy(weights, noise_scale=NOISE_SCALE):
    """Adds Gaussian noise to model weights to enforce differential privacy."""
    dp_weights = []
    for w in weights:
        noise = np.random.normal(loc=0.0, scale=noise_scale, size=w.shape)
        dp_weights.append(w + noise)
    return dp_weights

# Modified Federated Aggregation Loop with Privacy & Communication Tracking
local_models = [build_federated_model(X_train.shape[1]) for _ in range(NUM_CLIENTS)]
global_model = build_federated_model(X_train.shape[1])

NUM_ROUNDS = 5
for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n--- Federated Round {round_num}/{NUM_ROUNDS} (Differential Privacy Enabled) ---")
    global_weights = global_model.get_weights()
    
    local_weights_list = []
    round_payload_bytes = 0
    
    for i, (x_node, y_node) in enumerate(client_data):
        model = local_models[i]
        model.set_weights(global_weights)
        model.fit(x_node, y_node, epochs=3, batch_size=32, verbose=0)
        
        # Extract and apply Differential Privacy noise
        raw_weights = model.get_weights()
        private_weights = apply_differential_privacy(raw_weights)
        
        # Calculate communication payload size
        payload_bytes = sum(w.nbytes for w in private_weights)
        round_payload_bytes += payload_bytes
        
        local_weights_list.append(private_weights)
        print(f"  Farm Node {i+1} -> Differential Privacy Noise Injected | Update Payload: {payload_bytes / 1024:.2f} KB")
        
    # FedAvg Aggregation
    avg_weights = [np.mean(w_tuple, axis=0) for w_tuple in zip(*local_weights_list)]
    global_model.set_weights(avg_weights)
    print(f"Total Round Payload Exchanged: {round_payload_bytes / (1024 * 1024):.4f} MB")

global_model.save('models/saved_models/federated_global_model.h5')


--- Federated Round 1/5 (Differential Privacy Enabled) ---
  Farm Node 1 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
  Farm Node 2 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
  Farm Node 3 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
Total Round Payload Exchanged: 0.0806 MB

--- Federated Round 2/5 (Differential Privacy Enabled) ---
  Farm Node 1 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
  Farm Node 2 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
  Farm Node 3 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
Total Round Payload Exchanged: 0.0806 MB

--- Federated Round 3/5 (Differential Privacy Enabled) ---
  Farm Node 1 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
  Farm Node 2 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
  Farm Node 3 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
Total Round Payloa

  Farm Node 3 -> Differential Privacy Noise Injected | Update Payload: 27.51 KB
Total Round Payload Exchanged: 0.0806 MB


In [7]:
# Save as models/fusion_engine.py
import numpy as np

def compute_multimodal_crop_stress(leaf_disease_prob, env_stress_prob, w_leaf=0.6, w_env=0.4):
    """
    Combines leaf image disease confidence with tabular environmental risk 
    to generate an integrated crop stress index.
    """
    multimodal_score = (w_leaf * leaf_disease_prob) + (w_env * env_stress_prob)
    
    if multimodal_score >= 0.70:
        risk_level = "CRITICAL STRESS"
        action = "Immediate chemical/organic intervention and automated irrigation required."
    elif multimodal_score >= 0.40:
        risk_level = "MODERATE STRESS"
        action = "Increase field monitoring and balance soil nutrient levels."
    else:
        risk_level = "LOW RISK"
        action = "Crop health is optimal. Maintain current irrigation schedule."
        
    return {
        "multimodal_score": float(multimodal_score),
        "risk_level": risk_level,
        "recommended_action": action
    }